# Lecture 7: Information Extraction
### NLP Course 2027

---

## Learning Outcomes
- Understand the information extraction pipeline
- Apply keyphrase extraction techniques
- Build relation extractors using patterns and dependency parsing
- Evaluate NER and IE systems

**Primary References:** *NLP with Python* Ch.7 | *Practical NLP* Ch.5

## 1. Information Extraction Overview

Information Extraction (IE) converts **unstructured text** into **structured data**.

```
Unstructured:
  'Google acquired DeepMind for $500M in 2014.'
         ↓ IE Pipeline
Structured:
  {subject: 'Google', relation: 'acquired',
   object: 'DeepMind', amount: '$500M', year: '2014'}
```

### IE Pipeline
```
Raw Text
    ↓ Sentence Segmentation
    ↓ Tokenization
    ↓ POS Tagging
    ↓ Named Entity Recognition
    ↓ Relation Extraction
    ↓ Structured Output
```

### IE Tasks
| Task | Description |
|------|-------------|
| NER | Identify named entities |
| Relation Extraction | Find relationships between entities |
| Keyphrase Extraction | Find important phrases |
| Event Extraction | Find events and their arguments |
| Template Filling | Fill predefined schemas |
| Coreference Resolution | Resolve pronouns to entities |

In [1]:
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.corpus import treebank

nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

def run_ie_pipeline(text):
    """Run the full IE pipeline: tokenize → POS → NER."""
    sentences = nltk.sent_tokenize(text)
    results = []
    for sent in sentences:
        tokens = word_tokenize(sent)
        pos_tagged = pos_tag(tokens)
        ne_tagged = ne_chunk(pos_tagged)
        entities = [(ent.label(), ' '.join(w for w,t in ent.leaves()))
                    for ent in ne_tagged if hasattr(ent, 'label')]
        results.append({'sentence': sent, 'entities': entities})
    return results

text = """Barack Obama was born in Hawaii. He served as President of the United States.
Microsoft acquired LinkedIn for $26 billion in 2016."""

for r in run_ie_pipeline(text):
    print(f'Sentence: {r["sentence"]}')
    print(f'Entities: {r["entities"]}')
    print()

Sentence: Barack Obama was born in Hawaii.
Entities: [('PERSON', 'Barack'), ('PERSON', 'Obama'), ('GPE', 'Hawaii')]

Sentence: He served as President of the United States.
Entities: [('GPE', 'United States')]

Sentence: Microsoft acquired LinkedIn for $26 billion in 2016.
Entities: [('PERSON', 'Microsoft'), ('ORGANIZATION', 'LinkedIn')]



## 2. Relation Extraction

Relation extraction identifies **semantic relationships** between entities.

### Pattern-Based Extraction
```
Pattern: NP1 'acquired' NP2  →  ACQUISITION(NP1, NP2)
Pattern: NP1 'is a' NP2      →  IS_A(NP1, NP2)
Pattern: NP1 'was born in' NP2 → BORN_IN(NP1, NP2)
```

### Dependency-Based Extraction
Use dependency parse tree to find subject-verb-object (SVO) triples.

In [2]:
import re

def extract_relation_patterns(text):
    """Pattern-based relation extraction."""
    patterns = [
        (r'(.+?)\s+(?:acquired|bought|purchased)\s+(.+?)\s+for', 'ACQUISITION'),
        (r'(.+?)\s+(?:founded|created|established)\s+(.+)', 'FOUNDED'),
        (r'(.+?)\s+(?:is|are|was|were)\s+(?:a|an|the)\s+(.+)', 'IS_A'),
        (r'(.+?)\s+(?:born|raised)\s+in\s+(.+)', 'BORN_IN'),
        (r'(.+?)\s+(?:CEO|CTO|CFO|president|director)\s+of\s+(.+)', 'ROLE_IN'),
    ]
    found_relations = []
    sentences = nltk.sent_tokenize(text)
    for sent in sentences:
        for pattern, rel_type in patterns:
            matches = re.findall(pattern, sent, re.IGNORECASE)
            for match in matches:
                found_relations.append({'relation': rel_type,
                                        'args': [m.strip() for m in match],
                                        'source': sent[:60]})
    return found_relations

text = """
Mark Zuckerberg, CEO of Meta, founded Facebook in 2004.
Facebook was born in Cambridge, Massachusetts.
Microsoft acquired LinkedIn for $26.2 billion in 2016.
Satya Nadella is a software engineer and businessman.
"""
for rel in extract_relation_patterns(text):
    print(f"[{rel['relation']}] {rel['args']}")

[FOUNDED] ['Mark Zuckerberg, CEO of Meta,', 'Facebook in 2004.']
[ROLE_IN] ['Mark Zuckerberg,', 'Meta, founded Facebook in 2004.']
[BORN_IN] ['Facebook was', 'Cambridge, Massachusetts.']
[ACQUISITION] ['Microsoft', 'LinkedIn']
[IS_A] ['Satya Nadella', 'software engineer and businessman.']


## 3. Keyphrase Extraction

**Keyphrases** are words/phrases that summarize the main topics of a text.

### Methods
1. **TF-IDF based**: high score words in doc vs corpus
2. **TextRank**: graph-based, uses word co-occurrence
3. **RAKE** (Rapid Automatic Keyword Extraction): based on phrase boundaries
4. **YAKE**: statistical, unsupervised

### RAKE Algorithm
1. Split on stop words and sentence boundaries → candidate phrases
2. Score each word: `degree / frequency`
3. Score phrases: sum of word scores
4. Return top-scored phrases

In [3]:
from nltk.corpus import stopwords

def rake_keywords(text, max_words=3, top_n=10):
    """
    Simplified RAKE implementation.
    1) Split on stopwords/punctuation to get candidate phrases
    2) Score phrases by word frequency and co-occurrence
    """
    stop_words = set(stopwords.words('english'))
    # Step 1: Split text into candidate phrases
    words = text.lower()
    # Replace stopwords and punctuation with | delimiter
    import re
    stop_pattern = r'\b(?:' + '|'.join(re.escape(w) for w in stop_words) + r')\b'
    candidates_raw = re.sub(stop_pattern, '|', words)
    candidates_raw = re.sub(r'[^a-z|\s]', '|', candidates_raw)
    candidates = [phrase.strip() for phrase in candidates_raw.split('|')
                  if phrase.strip() and len(phrase.strip().split()) <= max_words]
    candidates = [c for c in candidates if c]

    # Step 2: Word frequency and degree (co-occurrence in phrases)
    from collections import defaultdict
    word_freq = defaultdict(int)
    word_degree = defaultdict(int)
    for phrase in candidates:
        phrase_words = phrase.split()
        for word in phrase_words:
            word_freq[word] += 1
            word_degree[word] += len(phrase_words) - 1

    # Step 3: Score each word
    word_score = {w: (word_degree[w] + word_freq[w]) / word_freq[w]
                  for w in word_freq}

    # Step 4: Score phrases
    phrase_scores = {}
    for phrase in set(candidates):
        score = sum(word_score.get(w, 0) for w in phrase.split())
        phrase_scores[phrase] = score

    return sorted(phrase_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

text = """
Natural language processing enables computers to understand human language.
Machine learning models are trained on large text corpora.
Transformer models like BERT have revolutionized NLP research.
Deep learning has enabled significant advances in language understanding.
"""
print('RAKE Keywords:')
for phrase, score in rake_keywords(text):
    print(f'  {score:.2f}  {phrase}')

RAKE Keywords:
  9.00  revolutionized nlp research
  9.00  large text corpora
  9.00  enabled significant advances
  8.50  machine learning models
  8.50  understand human language
  4.50  deep learning
  4.50  language understanding
  1.00  trained


In [4]:
# TF-IDF based keyphrase extraction
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def tfidf_keyphrases(docs, query_idx=0, top_n=10, ngram_range=(1,2)):
    """Extract keyphrases from a document using TF-IDF."""
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, stop_words='english',
                                  max_features=5000)
    tfidf_matrix = vectorizer.fit_transform(docs)
    feature_names = vectorizer.get_feature_names_out()
    doc_scores = tfidf_matrix[query_idx].toarray().flatten()
    top_indices = doc_scores.argsort()[-top_n:][::-1]
    return [(feature_names[i], doc_scores[i]) for i in top_indices if doc_scores[i] > 0]

docs = [
    "Machine learning models are trained using gradient descent optimization on large datasets.",
    "Natural language processing uses transformer architectures and attention mechanisms.",
    "Deep neural networks have layers of neurons with activation functions and backpropagation."
]
print('TF-IDF Keyphrases for doc 0:')
for phrase, score in tfidf_keyphrases(docs, query_idx=0):
    print(f'  {score:.4f}  {phrase}')

TF-IDF Keyphrases for doc 0:
  0.2294  using gradient
  0.2294  using
  0.2294  trained using
  0.2294  trained
  0.2294  optimization large
  0.2294  optimization
  0.2294  gradient
  0.2294  descent
  0.2294  descent optimization
  0.2294  learning models


In [5]:
# spaCy dependency-based SVO extraction
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')

    def extract_svo(text):
        doc = nlp(text)
        triples = []
        for token in doc:
            if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
                subjects = [t for t in token.children if 'subj' in t.dep_]
                objects  = [t for t in token.children if 'obj'  in t.dep_]
                for subj in subjects:
                    for obj in objects:
                        triple = (subj.text, token.lemma_, obj.text)
                        triples.append(triple)
        return triples

    test_sent = 'Google acquired DeepMind. Microsoft built Azure. Apple invented iPhone.'
    print('Subject-Verb-Object triples:')
    for s, v, o in extract_svo(test_sent):
        print(f'  ({s}, {v}, {o})')
except (ImportError, OSError):
    print('spaCy not available. Install: pip install spacy && python -m spacy download en_core_web_sm')

spaCy not available. Install: pip install spacy && python -m spacy download en_core_web_sm


## 4. IE Evaluation

Evaluating IE systems uses entity-level metrics:

```
Precision = TP / (TP + FP)   # of extracted entities, how many are correct?
Recall    = TP / (TP + FN)   # of true entities, how many did we find?
F1        = 2 * P * R / (P + R)
```

**Partial matching**: credit for partially correct extractions (e.g., 'Barack' instead of 'Barack Obama').

In [6]:
def evaluate_ner(predicted, gold):
    """
    Evaluate NER by comparing predicted vs gold entity sets.
    Each entity is a (text, type) tuple.
    """
    pred_set = set(predicted)
    gold_set = set(gold)
    tp = pred_set & gold_set
    fp = pred_set - gold_set
    fn = gold_set - pred_set
    precision = len(tp) / len(pred_set) if pred_set else 0
    recall    = len(tp) / len(gold_set) if gold_set else 0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
    print(f'TP: {len(tp)}  FP: {len(fp)}  FN: {len(fn)}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1:        {f1:.4f}')
    if fn:
        print(f'Missed: {fn}')
    return {'precision': precision, 'recall': recall, 'f1': f1}

gold = [('Google', 'ORG'), ('DeepMind', 'ORG'), ('London', 'GPE'), ('Demis Hassabis', 'PERSON')]
pred = [('Google', 'ORG'), ('DeepMind', 'ORG'), ('London', 'GPE'), ('Demis', 'PERSON')]
print('NER Evaluation:')
evaluate_ner(pred, gold)

NER Evaluation:
TP: 3  FP: 1  FN: 1
Precision: 0.7500
Recall:    0.7500
F1:        0.7500
Missed: {('Demis Hassabis', 'PERSON')}


{'precision': 0.75, 'recall': 0.75, 'f1': 0.75}

## Practice Exercises

See **`Lecture-07-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Task | Method | Output |
|------|--------|--------|
| NER | ne_chunk / spaCy | Entity spans + types |
| Relation Extraction | Patterns / dependency parse | (entity1, relation, entity2) |
| Keyphrase Extraction | RAKE / TF-IDF | Ranked phrases |
| Evaluation | Precision, Recall, F1 | Accuracy metrics |

**Next Lecture**: Real-World NLP Pipeline Design — from raw data to deployment.

---
*Book references: NLP with Python Ch.7 | Practical NLP Ch.5*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**